<a href="https://colab.research.google.com/github/BKBKlaassen/Gr8_ModelsForLanguageProcessing_assignments/blob/main/08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6: The AM Algebra and the AM Parser

Total points: 48

In this assignment you will complete code to implement an untyped version of the AM algebra, and perform an error analysis on some output of the AM parser.

There are 5 questions, 4 coding and one error analysis of AM parser output. **Be warned that this is a longer assignment, so plan accordingly.**

Please submit your answers to the coding questions in a copy of this Notebook. Name it `group_number.ipynb`, eg `03.ipynb`. Submit your answers to the last question as `group_number.pdf`.



## Overview

This Notebook is a combination tutorial and assignment. You are provided with a class for s-graphs, `SGraph`, and the explanations, examples, and exercises show you how it works. Use what you learn here, together with what you learned in class, to carry out the assignments.

You are also provided with modules defining trees and algebras, and these are inherited from in defining the `AMAlgebra` class and its terms. There are explanations, examples, and exerises explaining how algebra terms are implemented. Use this and what you learned in class to answer the question about terms.

Finally, you will do some error analysis on AM Parser output.

# <font color="red">Contributions</font>

Group number: 8

Group members: Bjorn Klaassen, Noah de Jonge

Who contributed to which exercises (you don't need to be very detailed): Assignments done together

## Part 0

Install dependencies

In [51]:
!pip install python-socketio eventlet penman amconll wikipedia conllu

In [52]:
# use on Colab or as an alternative to the instructions below for installing Vulcan
# ! rm -rf ./vulcan
! git clone https://github.com/jgroschwitz/vulcan.git
!pip install ./vulcan

fatal: destination path 'vulcan' already exists and is not an empty directory.
Processing ./vulcan
  Preparing metadata (setup.py) ... done
  Created wheel for vulcan: filename=vulcan-0.0.1-py3-none-any.whl size=51485 sha256=20b97a0d47461133fae8891a4df31a2f9b36ae4e88aec1e112eaefc9f1b40a1b
  Stored in directory: /tmp/pip-ephem-wheel-cache-5ucj_y6c/wheels/70/92/0b/d48d28e8ff0670e6aa0fcb9fed66aa2fac2907306104bfd8d3
Successfully built vulcan
  Attempting uninstall: vulcan
    Found existing installation: vulcan 0.0.1
    Uninstalling vulcan-0.0.1:
      Successfully uninstalled vulcan-0.0.1


In [53]:
# assignment files
# ! rm -rf assigntools
! git clone https://github.com/megodoonch/assigntools.git

fatal: destination path 'assigntools' already exists and is not an empty directory.


In [54]:
import copy
import penman
import sys
import nltk
import logging
from copy import deepcopy
from typing import Set, List, Dict

logger = logging.getLogger(__name__)
# logger.setLevel(logging.DEBUG)  # Switch to DEBUG for more elaborate printing in some functions
# logger.setLevel(logging.INFO)
logger.setLevel(logging.WARNING)


In [55]:
# import the provided modules
from assigntools.M4LP.HW5.graphs import SGraph, Graph
from assigntools.M4LP.HW5.algebra import AlgebraOp, AlgebraTerm, AlgebraError, Algebra

## Part 1: Get Vulcan working on your own computer

Vulcan is a visualisation tool for linguistic structures. You can use it to visualise the graphs you build in Part 2, and in Part 3 you will use it to do error analysis of AM parser output.

1. Go to [https://github.com/jgroschwitz/vulcan](https://github.com/jgroschwitz/vulcan) and download the repository. If you click on the green "Code" button, you can choose to download as a Zip file, or to copy the location so that you can use Git to clone it. If you downloaded it as a zip file, make sure you unzip it.
2. Follow the instructions on the github page to get started. You may want a new virtual environment (e.g. `conda create -n "M4LP"`), but in any case, once you're ready, use `pip` to install the dependencies.
3. In a terminal, navigate to the Vulcan folder, and test your installation by running the following command. (Your computer might want you to use `python3` instead, which is also fine.)

```commandline
python launch_vulcan.py little_prince.pickle
```

It should open up a browser window. If not, it should print a URL that you can copy and paste into a browser window. You should see trees, sentences, and AMRs for The Little Prince.

You should be able to navigate with the buttons at the top. There is also a search function.
```

In [56]:
# import useful things from Vulcan
from vulcan.pickle_builder.pickle_builder import PickleBuilder
from vulcan.data_handling.format_names import FORMAT_NAME_GRAPH, FORMAT_NAME_STRING, FORMAT_NAME_NLTK_TREE

These function is provided to help you use Vulcan to visualise your graphs and compare them to the correct graphs. Any pickles created with this function can be read with the `launch_vulcan.py` script. Run the `help` function to learn more about them. You can see the source code in `assigntools/M4LP/HW5`.

You can't launch vulcan on Google Colab. If you make pickles of your outputs, download the pickles and open them on your own computer, e.g.

```commandline
python launch_vulcan.py path/to/my/downloaded/pickle.pickle
```

In [57]:
from assigntools.M4LP.HW5.vulcan_pickles import create_vulcan_pickle_gold_and_student_graphs
from assigntools.M4LP.HW5.vulcan_pickles import create_vulcan_pickle_of_graphs
from assigntools.M4LP.HW5.vulcan_pickles import create_vulcan_pickle_terms_and_gold_graphs_and_student_graphs
from assigntools.M4LP.HW5.vulcan_pickles import create_vulcan_pickle_terms_and_graphs

In [58]:
help(create_vulcan_pickle_gold_and_student_graphs)

Help on function create_vulcan_pickle_gold_and_student_graphs in module assigntools.M4LP.HW5.vulcan_pickles:

create_vulcan_pickle_gold_and_student_graphs(gold_graphs, student_graphs, pickle_path, comments=None)
    Creates a vulcan-readable pickle comparing the correct graphs to your own graphs.
    If there is an error converting one of your graphs to a penman.Graph,
        a graph with one node labelled with the error is created instead.
    :param gold_graphs: list of correct SGraphs
    :param student_graphs: list of students' SGraphs
    :param pickle_path: path to write the pickle to (including pickle name)
    :param comments: optional list of strings, one for each graph pair.



## Part 2: Graphs

Follow along with the cells below. There are demos, exercises (not for points), and assignments (marked; for points.)

There is a `Graph` class implemented for you. You can see the source code in `assigntools/M4LP/HW5`, and you can get information about the class and its methods with the `help` funtion


In [59]:
help(Graph)

Help on class Graph in module assigntools.M4LP.HW5.graphs:

class Graph(builtins.object)
 |  Graph(nodes: Set = None, edges: Dict = None, node_labels: Dict = None, root=None)
 |
 |  Attributes:
 |      nodes: set
 |      edges: dict from origin nodes to lists of (target node, edge label) or target if unlabeled
 |      node_labels: dict from node to label
 |      root: specially marked node
 |
 |  Methods defined here:
 |
 |  __add__(self, other)
 |      Adds two graphs together, keeping the root at the root of self,
 |      and otherwise simply taking the union of the nodes, edges, and labels.
 |      :param other: Graph
 |      :return: Graph
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __init__(self, nodes: Set = None, edges: Dict = None, node_labels: Dict = None, root=None)
 |      Initialise a Graph
 |      Args:
 |          nodes: a set; if None, default is an empty graph with no nodes.
 |          edges: a dict from nodes to lists of nodes or (node, str) pairs. T

A unary, unlabelled graph can be defined as follows:

In [60]:
g_over_int = Graph(nodes={0})
g_over_str = Graph(nodes={"a"})

In [61]:
# basic print
print(g_over_int)

# as a graphviz/dot input
print("\nDot:")
print(g_over_int.to_graphviz())

# as a penman.Graph object
print("\nPenman:")
print(g_over_int.to_penman())


nodes:	{0}


Dot:
digraph g {
0;
}

Penman:
Graph(
  [],
  epidata={})


A labelled, rooted, unary graph can be defined as follows:

In [62]:
cat = Graph(nodes={0}, node_labels={0: "cat"}, root=0)

In [63]:
# basic print
print(cat)

# as a graphviz/dot input
print("\nDot:")
print(cat.to_graphviz())

# as a penman.Graph object
print("\nPenman:")
print(cat.to_penman())

rt:	0
nodes:	{0}
labels:	{0: 'cat'}


Dot:
digraph g {
0 [style=bold, label="cat"];
}

Penman:
Graph(
  [('0', ':instance', 'cat')],
  epidata={})


A graph with edges:

In [64]:
# make sure the values of `edges` are LISTS of (target, label) pairs, since a node can have multiple outgoing edges.
cat_slept = Graph(nodes={0,1}, node_labels={0: "sleep-01", 1: "cat"},
                   edges={0: [(1, "ARG0")]}, root=0)

# basic print
print(cat_slept)

# as a graphviz/dot input
print("Dot:")
print(cat_slept.to_graphviz())

# as a penman.Graph object
print("\nPenman:")
print(cat_slept.to_penman())


rt:	0
nodes:	{0, 1}
labels:	{0: 'sleep-01', 1: 'cat'}
edges:
	 0 -ARG0-> 1

Dot:
digraph g {
0 [style=bold, label="sleep-01"];
1 [label="cat"];
0->1 [label=ARG0];
}

Penman:
Graph(
  [('0', ':instance', 'sleep-01'),
   ('1', ':instance', 'cat'),
   ('0', ':ARG0', '1')],
  epidata={})


In [65]:
graphs = [cat, cat_slept]
penman_graphs = [g.to_penman() for g in graphs]

# penman.dump can write a standard AMR file. If you make the second argument sys.stdout, you can see it here.
penman.dump(penman_graphs, sys.stdout)

# write to file
penman.dump(penman_graphs, "amrs.txt")


(0 / cat)

(0 / sleep-01
   :ARG0 (1 / cat))


You can visualise an AMR corpus with Vulcan using a different script from the one you tested out above, which is for pickles. To try this, download amrs.txt to your computer and from the vulcan folder, run:

```commandline
python visualize_amr_corpus.py path/to/amrs.txt
```

For example, on my computer I used:

```commandline
python visualize_amr_corpus.py ../M4LP/HW5/amrs.txt
```

Notice at the bottom, it says "sentence not found in corpus". This just means the AMR corpus we wrote to amrs.txt doesn't include a sentence for the graph. If you want to add that, you can.

In [66]:
# a penman graph can have metadata, which is stored as a dict in graph.metadata.
# the key is the name of the metadata entry, and the value is what should be printed with it.
cat_penman = cat.to_penman()
cat_penman.metadata["snt"] = "cat"

cat_slept_penman = cat_slept.to_penman()
cat_slept_penman.metadata["snt"] = "the cat slept"

# standardly, AMR sentences are annotated with "snt", so Vulcan can read this new file with the sentences.
penman_graphs = [cat_penman, cat_slept_penman]
penman.dump(penman_graphs, sys.stdout)
penman.dump(penman_graphs, "amrs.txt")



# ::snt cat
(0 / cat)

# ::snt the cat slept
(0 / sleep-01
   :ARG0 (1 / cat))


You'll need to stop (CTRL-c) and restart (up, enter, refresh page) the Vulcan server to see the updated corpus.

### QUESTION 1: AM Algebra Types [10 points]

Recall that in the AM Algebra, the *type* of an Annotated S-Graph is itself a graph. While the s-graph that we put together to build the meaning representation are labelled s-graph, their types are very simple graphs. They are unlabelled graph whose nodes ARE sources. That is, the nodes are just strings that happen to be the same as sources in the s-graph.

The partially-completed class below, `AMType`, inherits from `Graph`. It implements these type graphs. An example of the classic control verb type is instantiated below.

Your job is to implement the `request` method. Use the definition of *the request in graph G at source X* given in the lectures. (The definition in the advanced reading from the thesis also includes edge labells for renaming, which we won't use, but is otherwise the same.) See your pervious assignment for the definition of a *subgraph generated by a node*.

Hint 1: You'll probably find some methods of the `Graph` class useful

Hint 2: You may want to implement a recursive function within a function to recursively extract the nodes and edges. This can be done in Python as follows:

In [67]:
def my_recursive_function(d, match):
    """
    Find all keys in a recursive dict structure that match something in match
    """
    matching_keys = []  # fill this list

    def recursive_inner_function(d):
        for k in d:
            if k in match:
                # update the list
                matching_keys.append(k)
            # recursive function call on sub-dict
            recursive_inner_function(d[k])

    # Start with the full dict
    recursive_inner_function(d)
    # return filled list
    return matching_keys


my_dict = {1: {2: {3: {}, 4: {5: {}}}, 4: {2: {}}}}
search = [2,4,6]

my_recursive_function(my_dict, search)


[2, 4, 4, 2]

In [73]:
# %%%%%%%%%%%%%%%% QUESTION 1 %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


class AMType(Graph):
    """
    A class for the graph definition of types of annotated s-graphs
    Used for the AM algebra.

    The graph is unlabelled, and the nodes are themselves sources.

    Attributes:
        nodes: set of strings: these are sources,
            usually from the s-graph it's the type of.
        edges: unlabelled edges.
    """
    def __init__(self, nodes: Set[str], edges: dict[str: List[str]] or None=None):
        super().__init__(nodes, edges)

    def request(self, node):
        """
        Returns a new AMType, which is the request in this graph at node
        Used to calculate whether Apply is permitted
        A request at a node is the way to check wether a certain apply
        or modify operation is permitted for succesfully merging graphs,
        so as you request for a specific node, you get what sources
        should be fufilled by the compliment of the incoming graph.
        """
        new_edges = {}
        for i in self.edges:
          if i == node:
            continue
          key = i
          value = [n for n in self.edges[key] if n != node]
          new_edges[key] = value

        if len(new_edges) == 0:
          new_edges = None

        new_nodes = set([n for n in self.nodes if n != node])
        return AMType(new_nodes,new_edges)

    def __repr__(self):
        edgeless = self.nodes - self.edges.keys()
        for n in self.edges:
            edgeless = edgeless - set(self.get_targets(n))
        ret = "["
        ret += " ".join(edgeless)
        edges = []
        for origin in self.edges:
            for target in self.edges[origin]:
                edges.append( f"{origin}->{target}")
        edge_string = "\n".join(edges)
        ret += edge_string
        ret += "]"
        return ret



In [74]:
# Example: subject control verb (e.g. "want")
control_type = AMType({"O", "S"}, {"O": ["S"]})

print(control_type)

# intransitive verb type
intrans_type = AMType({"S"})
print(intrans_type)

[O->S]
[S]


In [75]:
# test
assert control_type.request("O") == intrans_type, f"\n{control_type.request("O")} != \n{intrans_type}"

Unmatched parenthesis at position 5 in processing (None)
Unmatched parenthesis at position 5 in processing (None)
ERROR:assigntools.M4LP.HW5.graphs:graphs [S] and [S] can't be compared due to error 'NoneType' object has no attribute 'rename_node'


AssertionError: 
[S] != 
[S]

In [ ]:
coord_control_type = AMType({"op1", "op2", "O", "S"}, {"op1": ["O"], "op2": ["O"], "O": ["S"]})
print(coord_control_type)

### QUESTION 2: Test `request` [1 point]

Write a test for your `request` method by comparing the request in `coord_control_type` at `"op2"` to a hard-coded AMType for the correct request graph.

In [ ]:
# %%%%%%%%%%%%%%%% QUESTION 2 %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# YOUR TEST HERE


### Part 3: SGraphs

**Throughout this assignment, use this global variable `RT` for all root sources**

In [ ]:
# Root source definition
RT = "rt"

`graphs.py` also defines the class `SGraph`, which implements s-graphs (graphs with sources), as defined in the HR algebra. (Courcelle, B. (1993). Graph grammars, monadic second-order logic and the theory of graph minors. In
N. Robertson and P. Seymour (Eds.), Graph Structure Theory, pp. 565—590. AMS.)

`SGraph` inherits from `Graph`.

We can see its properties with the `help` function.

In [ ]:
help(SGraph)

A graph with sources. By convention, when we print, we write source `X` as `<X>` to distinguish it from a regular label.

In [ ]:
slept = SGraph(nodes={0,1}, node_labels={0: "sleep-01"}, edges={0: [(1, "ARG0")]}, root=0, sources={"S": 1, RT: 0})

print("slept:")
print(slept)
print("Dot:")
print(slept.to_graphviz())
print("Penman:")
print(slept.to_penman())

slept_without_source = SGraph(nodes={0,1}, node_labels={0: "sleep-01"}, edges={0: [(1, "ARG0")]}, root=0)
print("\nwithout source: equal?")
print(slept_without_source == slept)


# Let's also add a root-source to cat_slept
cat_slept = SGraph(nodes={0,1}, node_labels={0: "sleep-01", 1: "cat"},
                   edges={0: [(1, "ARG0")]}, root=0, sources={RT: 0})


The fact that the `__add__` method is implemented for the `SGraph` class means we can actually add two graphs with `+`. This implements the HR algebra operation Merge (`||`), so any common sources are merged. Because we are interested in rooted graphs, if both graph have a root, + makes the root of the first graph the root of the resulting graph.

In [ ]:
dream = SGraph(nodes={0,1}, node_labels={0: "dream-01"}, edges={0: [(1, "ARG0")]}, root=0, sources={"S": 1})

sleep_and_dream = slept + dream

# notice that even though sleep and dream both only had nodes 0 and 1, this new graph also has node 2.
print(sleep_and_dream)

In [ ]:
# Add this to the corpus

sleep_and_dream_penman = sleep_and_dream.to_penman()
sleep_and_dream_penman.metadata["snt"] = "slept and dreamt"

penman_graphs.append(sleep_and_dream_penman)

penman.dump(penman_graphs, "amrs.txt")

Re-download the corpus, and restart the Vulcan server and refresh the browser window to see the resulting graph. Notice how the `<S>` source is common to the dream and sleep predicates.

## Exercise (not to hand in):

`dream` has no root source, but `slept` does. What happens if `dream` has a root source at `0`? Why would we want it to have a root source at its root? Why do we want the `SGraph` code to raise an error when `dream` with a root-source is merged with `slept` with a root-source?

### Exercise (not to hand in):

Make a new SGraph `cat_s` that's like `cat` except its root is an `"S"`-source.  Add the new `cat` graph and the `slept` graph together with `+` in such a way that the root is at the `sleep-01` node (not the `cat` node). You should find that the graph is isomorphic to the one defined in `cat_slept_s` below. Check they are indeed equal.

Note: checking isomophism between graphs is non-trivial, because there are so many possible node- and edge-matching to try. Equality in SGraph uses Smatch, which approximates isomorphism. There is always a chance it will report that two graphs are not isomorphic even when they are.

In [ ]:
# # EXERCISE (for practice, not to hand in)
# # Your code:

cat = SGraph.from_graph(cat, "S")
cat_slept = slept + cat

# cat_slept = ...

# this is the right graph. Equality checks isomorphism, so checking equality with your graph and this one should print True
cat_slept_s = SGraph(nodes={0, 1}, edges={0: [(1, 'ARG0')]}, node_labels={0: 'sleep-01', 1: 'cat'}, sources={"S": 1, RT:0}, root=0)
print(cat_slept_s)

assert(cat_slept == cat_slept_s)

# # You can compare the two with Vulcan by making this pickle and opening it with python launch_vulcan.py path/to/cat_slept.pickle
create_vulcan_pickle_gold_and_student_graphs([cat_slept_s], [cat_slept], "cat_slept.pickle")


The HR algebra also has operations for changing sources. This is implemented as methods `rename` and `forget` of `SGraph`.

In [ ]:
# forget a source (remove the source, but keep the node)
print(cat_slept_s)
cat_slept_s.forget("S")
print(cat_slept_s)

# removing the S-source makes this equal to cat_slept again.
print(cat_slept_s == cat_slept)


`rename` renames a source from A to B. That is, the A-source stops being an A-source and becomes a B-source instead.

In [ ]:
# rename S to O
slept.rename("S", "O")
print(slept)

slept.rename("O", "S")
print(slept)


### Exercise (not to be handed in):

Use a combination of `+`, `rename`, and `forget` to put together `cat_o` and `slept` in such a way that you end up with a graph isomorphic to `cat_slept`:

In [ ]:
# # EXERCISE
slept = SGraph(nodes={0,1}, node_labels={0: "sleep-01"}, edges={0: [(1, "ARG0")]}, root=0, sources={"S": 1, RT: 0})

cat_o = SGraph({0}, node_labels={0: "cat"}, sources={"O": 0}, root=0)

cat_slept = SGraph(nodes={0, 1}, edges={0: [(1, 'ARG0')]}, node_labels={0: 'sleep-01', 1: 'cat'}, sources={RT:0}, root=0)


# Your code

cat_o.rename("O", "S")
final_graph = slept + cat_o
final_graph.forget("S")

# final_graph = ...

print(final_graph)

assert(final_graph == cat_slept), f"{final_graph} \n!=\n {cat_slept}"

# # If you want to see each step, you can copy and save each intermediate graph in its own variable name and make a pickle
# # create_vulcan_pickle_of_graphs(list_of_intermediate_graphs)
# create_vulcan_pickle_of_graphs([cat_s, cat_o, slept, my_graph, final_graph], "building_cat_slept.pickle", ["cat_s", "cat_o", "slept", "put together", "forget O"])

## Part 4: UnTyped AM Algebra

### QUESTION 3 [12 points]

The class below, `AMAlgebra`, defines an **untyped** variant of the AM Algebra you learned about in class. This means the objects of the algebra as `SGraph` objects, not as-graphs (annotated s-graphs, i.e. s-graphs annotated with their types.) Therefore `apply` and `modify` are defined on most pairs of graphs.

Your job is to complete the `AMAlgebra` class by implementing the `apply` and `modify` methods of the class so that they correctly implement the algebra operations Apply and Modify, but without the type checking. Use the docstrings, the slides, and the paper to help you. Define them in terms of the HR algebra operations. Use the global root source `RT` for roots.

Don't change the signatures of the methods (or if you do, make any additions optional so the tests still work), and make sure you are returning an `SGraph`.

Apply and Modify as Algebra Operations are binary, and there is one for each source you might want to use. This functionality is built into the algebra, and so the `AlgebraOp` objects are in fact binary. However, they make use of the class methods `apply` and `modify` that you are building, and as you can see from the provided signatures, they both take three arguments: the source and the two graphs. You will be able to see the resulting binary algebra operations in play in the tests provided below.

Warning: you may find you need to copy one or both graphs in `apply` and `modify`. `+` is already defined to copy the graphs, but `rename` and `forget` apply in-place.

In [ ]:
# %%%%%%%%%%%%%%%% QUESTION 3 %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


class AMAlgebra(Algebra):
    """
    A typeless AM algebra.
    Apply and Modify work as long as the underlying operations work, otherwise they throw the underlying errors.
    Domain: SGraphs.
    Binary operations: Apply and Modify (by source).
    Attributes:
        zero: empty graph.
        domain_type: the type SGraph, or a subclass thereof.
        sources: list of strings (could be anything) for the non-root source of the SGraphs.
                        (default S, O, O2-O8, OP1-OP20, M, A)
        root_source: str: root source name (default RT = "rt")
        app_only: sources that are only used in APP.
        mod_only: sources that are only used in MOD.
        ops: dict of operation names to AlgebraOps. Traditionally just the non-constants
        constant_maker: a function that makes constant AlgebraOps.
         For SGraphs you probably want to add this after initialisation by adding a dict of constants,
         which makes the constant maker a look-up function.

    """

    def __init__(self, sources=None, app_only=None, mod_only=None,
                 graph_type=SGraph, root_source=None):
        """
        Initialises an instance of the AMAlgebra.

        :param sources: list[str]: the sources of the algebra.
            If None, default of M, A, S, O, O2, ... O8, op1, ..., op20 is used.
        :param app_only: sources that can only have an Apply operation. Default all opi.
        :param mod_only: sources that can only have a Modify operation. Default M, A.
        :param graph_type: the type of the class of graph. Must be subclass of SGraph. Default SGraph.
        :root_source: str: root source name, stored as self.root_source. Default global RT.
        """
        super().__init__(name="AM algebra", domain_type=graph_type)
        self.add_constant_maker()
        print("added constant maker")

        self.OP = "OP"
        self.root_source = RT if root_source is None else root_source

        # some AMR defaults
        if sources is None:
            sources = ["M", "S", "O", "A"]
            for i in range(2, 8):
                sources.append(f"O{i}")
            for i in range(1, 20):
                sources.append(f"{self.OP}{i}")
            if app_only is None:
                app_only = [source for source in sources if source.startswith(self.OP)]
            if mod_only is None:
                mod_only = ["M", "A"]

        self.sources = sources
        self.app_only = app_only
        self.mod_only = mod_only

        self.ops = {}

        # all Algebra operations defined here
        for s in [x for x in self.sources if x not in self.mod_only]:
            function = self.make_apply_operation(s)
            name = f"App_{s}"
            op = AlgebraOp(name, function)
            self.ops[op.name] = op
        for s in [x for x in self.sources if x not in self.app_only]:
            function = self.make_modify_operation(s)
            name = f"Mod_{s}"
            op = AlgebraOp(name, function)
            self.ops[op.name] = op

    def __repr__(self):
        return self.name

    def ops_repr(self):
        """
        For printing the operations
        """
        apps = []
        mods = []
        other = []
        for op in self.ops:
            if op[:3].lower() == "app":
                apps.append(op)
            elif op[:3].lower() == "mod":
                mods.append(op)
            else:
                other.append(op)
        ret = "\n\tApp:\n\t\t"
        for op in sorted(apps):
            ret += f" {op}"
        ret += "\n\tMod:\n\t\t"
        for op in sorted(mods):
            ret += f" {op}"
        return ret

    def make_apply_operation(self, source):
        def apply_to_list(args):
            head = args[0]
            argument = args[1]
            return self.apply(source, head, argument)

        return apply_to_list

    def make_modify_operation(self, source):
        def modify_source(args):
            head = args[0]
            modifier = args[1]
            return self.modify(source, head, modifier)

        return modify_source

    def make_leaf(self, name, function=None):
        """
        Given a constant name, looks up the constant in self.constants.
        If not found, makes a unary node with the label.
        returns an AlgebraTerm that evaluates to that graph.
        """
        return AlgebraTerm(self.constant_maker(name))

    def default_constant_maker(self, word, label=None):
        """
        Makes unary graph with word as label
        """
        return AlgebraOp(word, self.domain_type({0}, node_labels={0: word}, root=0, sources={self.root_source: 0}))

    def apply(self, source: str, head: SGraph, argument: SGraph):
        """
        Implements the AM Apply operation, with no typing.
        # TODO update this docstring with more info about your function

        @param source: str: the source to put the root of argument into.
        @param head: the head and functor of the operation (the one with the relevant source).
        @param argument: the argument (the one to be added to head).
        @return: SGraph: the result of Apply_source(head, argument).
        """
        # check types and that the source is actually present
        assert isinstance(head, self.domain_type), f"head must be an {self.domain_type} but is a {type(head)}"
        assert isinstance(argument, self.domain_type), f"argument must be an SGraph but is a {type(head)}"

        # TODO Check that the source is present
        # TODO Finish this function
        ...


    def modify(self, source: str, head: SGraph, modifier: SGraph):
        """
        implements the AM Modify operation, with no typing
        # TODO: update this docstring with more info about your function
        @param source: str: the source to put the root of argument into
        @param head: the graph to have a modifier added to. Will keep this one's root.
        @param modifier: the modifier to be added (functor; i.e. the one with the relevant source)
        @return: SGraph
        """
        assert isinstance(head, self.domain_type), f"head must be an SGraph but is a {type(head)}"
        assert isinstance(modifier, self.domain_type), f"modifier must be an SGraph but is a {type(head)}"

        # TODO Check that the source is present
        # TODO Finish this function
        ...

## Tests and troubleshooting

Below you will find some tests, as well as code to help visualise output so you can see how your code is working on a range of inputs. You will probably want to add some more tests for yourself.

Please note that when it is graded, your code will be tested on additional examples.

In [ ]:
# Some vocabulary

like = SGraph(
    {0, 1, 2},
    {
        1: [(0, "ARG0"), (2, "ARG1")]
    },
    {1: "like-01"},
    {"S": 0, "O": 2, RT: 1},
    1
)

sleep = SGraph(
    {0, 1},
    {
        1: [(0, "ARG0")]
    },
    {1: "sleep-01"},
    {"S": 0, RT: 1},
    1
)

arrived = SGraph(
    {0, 1},
    {
        1: [(0, "ARG0")]
    },
    {1: "arrive-01"},
    {"S": 0, RT: 1},
    1
)


tried = SGraph(
    {0, 1, 2},
    {
        1: [(0, "ARG0"), (2, "ARG1")]
    },
    {1: "try-01"},
    {"S": 0, "O": 2, RT: 1},
    1
)

mary = SGraph(
    {0, 1, 2},
    edges={0: [(1, "name")],
           1: [(2, "op1")]},
    node_labels={0: "person", 1: "name", 2: "Mary"},
    sources={RT: 0},
    root=0
)

cat = SGraph(
    {0},
    node_labels={0: "cat"},
    sources = {RT: 0},
    root=0
)

whistling = SGraph(
    {0, 1, 2},
    {
        1: [(2, "ARG0")],
        0: [(1, "manner")]
    },
    {1: "whistle-01"},
    {"S": 2, "M": 0, RT: 1},
    1
)

cute = SGraph({0, 1},
              node_labels={0: "cute"},
              edges={1: [(0, "mod")]},
              root=0,
              sources={"M": 1, RT: 0})

vocabulary = [mary, cat, tried, like, whistling, sleep, cute, arrived]




In [ ]:
# Use the S-graphs defined above to create S-graph constants for the untyped AM-Algebra

# these are the zero-ary symbols of the signature
names = ["mary", "cat", "tried", "like", "whistling", "sleep", "cute", "arrived"]

# an AlgebraOp object links the symbol to its interpretation
constants = {}
for name, g in zip(names, vocabulary):
    constants[name] = AlgebraOp(name, g)



### Create an algebra

Run the cell below to build an AM-Algebra with the graph constants defined in the cell above.

In [ ]:
am_algebra = AMAlgebra()
am_algebra.add_constants(constants)

In [ ]:
print("Graph constants:\n")
for name, op in am_algebra.constants.items():
    print(name, ":\n", op.function)

In [ ]:
# try out Apply method

print("AppO(like, cat)")
like_cat = am_algebra.apply("O", like, cat)
print(like_cat)

print("AppO(tried, like_cat)")
tried_like_cat = am_algebra.apply("O", tried, like_cat)
print(tried_like_cat)

print("AppS(tried_like_cat, mary)")
full = am_algebra.apply("S", tried_like_cat, mary)
print(full)




In [ ]:
# Try modify method

print("ModM(cat, cute)")
print(cute)
cute_cat = am_algebra.modify("M", cat, cute)
print(cute_cat)

print("ModM(arrived, whistling)")
print(arrived)
arrived_whistling = am_algebra.modify("M", arrived, whistling)
print(arrived_whistling)

In [ ]:
# Run this cell to visualise the constants and graphs above

graphs = []
names = []
for name, g in am_algebra.constants.items():
    graphs.append(g.function)
    names.append(name)

graphs.append(like_cat)
names.append("like cat")

graphs.append(tried_like_cat)
names.append("tried like cat")

graphs.append(full)
names.append("full")

graphs.append(cute_cat)
names.append("cute cat")

graphs.append(arrived_whistling)
names.append("arrived whistling")

create_vulcan_pickle_of_graphs(graphs, "apply.pickle", names)

### Algebra Terms

Since the AM Algebra is an algebra, we can define terms over the operation names and constant names, and if we evaluate the terms, we get `SGraph`s. A term is just a tree with operation names as internal nodes and constant names as leaves.

Vulcan can also visualise the terms and output graphs.

In [ ]:
# As Algebra Terms

# for convenience, let's store some common AlgebraOps in variables
app_o = am_algebra.ops["App_O"]
app_s = am_algebra.ops["App_S"]
mod_m = am_algebra.ops["Mod_M"]

# a leaf term can be made with the make_leaf method of the algebra. If the string you give it is in the constants dict in the algebra, it will use that constant, and otherwise will create a graph that's just a single node with that label.
like_term = am_algebra.make_leaf("like")
print(like_term, like_term.evaluate())
dog_term = am_algebra.make_leaf("dog")
print("new constant:", dog_term, dog_term.evaluate())

# you can also make a term by hand if you want to tell it to use a particular graph.
# Make it an AlgebraTerm, and the parent (well, only node!) of the term is an AlgebraOp with a name and an SGraph.
sing = SGraph(nodes={0, 1}, edges={1: [(0, 'ARG0')]}, node_labels={1: 'sing'}, sources={'S': 0, RT: 1}, root=1)
sing_term = AlgebraTerm(AlgebraOp("sing", sing))
print(sing_term, sing_term.evaluate())
print("sing term evaluates to sing graph:", sing_term.evaluate() == sing)



In [ ]:
# Putting graphs together

# This is the algebra term that builds the "like cat" SGraph, and if you evaluate it, you get that SGraph.
like_cat_term = AlgebraTerm(app_o, [am_algebra.make_leaf("like"), am_algebra.make_leaf("cat")])
print("tree:", like_cat_term)

# this doesn't work on CoLab, but it should work at home to display the term.
# like_cat_term.to_nltk_tree().draw()
Algebra
print("graph:", like_cat_term.evaluate())

# gold graph
like_cat_graph = SGraph(nodes={0, 1, 2}, edges={1: [(0, 'ARG0'), (2, 'ARG1')]},
                        node_labels={1: 'like-01', 2: 'cat'}, sources={'S': 0, RT: 1}, root=1)
print(like_cat_term.evaluate() == like_cat_graph)

In [ ]:
# The full graph meaning something like "Mary tried to like the cat"

mary_cat_term = AlgebraTerm(app_s, [AlgebraTerm(app_o, [am_algebra.make_leaf("tried"), like_cat_term]), am_algebra.make_leaf("mary")])

# gold graph
mary_cat_graph = SGraph(nodes={0, 1, 2, 5, 7, 8},
                        edges={1: [(0, 'ARG0'), (2, 'ARG1')], 2: [(0, 'ARG0'), (5, 'ARG1')], 7: [(8, 'op1')], 0: [(7, 'name')]},
                        node_labels={1: 'try-01', 5: 'cat', 2: 'like-01', 7: 'name', 8: 'Mary', 0: 'person'},
                        sources={RT: 1}, root=1)

my_graph = mary_cat_term.evaluate()
print(my_graph == mary_cat_graph)


print(mary_cat_term)

In [ ]:
# Cute cat (Modify)

cute_cat_term = AlgebraTerm(mod_m, [am_algebra.make_leaf("cat"), am_algebra.make_leaf("cute")])

print(am_algebra.make_leaf("cute").evaluate())

# gold graph
cute_cat_graph = SGraph(nodes={0, 2}, edges={0: [(2, 'mod')]}, node_labels={0: 'cat', 2: 'cute'}, sources={RT:0}, root=0)

# your graph, minus the root source if you have one
my_graph = cute_cat_term.evaluate()

print(my_graph == cute_cat_graph)

In [ ]:
# Apply and modify together

mary_arrived_whistling_term = AlgebraTerm(app_s, [AlgebraTerm(mod_m, [am_algebra.make_leaf("arrived"), am_algebra.make_leaf("whistling")]), am_algebra.make_leaf("mary")])

print(mary_arrived_whistling_term.evaluate())

# gold graph
mary_arrived_whistling_graph = SGraph(nodes={0, 1, 4, 6, 7},
                                      edges={1: [(0, 'ARG0'), (4, 'manner')], 4: [(0, 'ARG0')], 6: [(7, 'op1')], 0: [(6, 'name')]},
                                      node_labels={1: 'arrive-01', 4: 'whistle-01', 6: 'name', 7: 'Mary', 0: 'person'},
                                      sources={RT: 1}, root=1)


# your graph, minus the root source if you have one
my_graph = mary_arrived_whistling_term.evaluate()

print(my_graph == mary_arrived_whistling_graph)


Put any terms you'd like to see into a pickle. You can add comments, for instance the sentence under `comments`.

In [ ]:
# Without the gold graphs
create_vulcan_pickle_terms_and_graphs([mary_arrived_whistling_term, mary_cat_term],
                                      "terms.pickle",
                                      comments=["Mary arrived whistling", "Mary tried to like the cat"])

In [ ]:
# With the gold graphs
create_vulcan_pickle_terms_and_gold_graphs_and_student_graphs([mary_arrived_whistling_term, mary_cat_term, cute_cat_term, like_cat_term],
                                                              [mary_arrived_whistling_graph, mary_cat_graph, cute_cat_graph, like_cat_graph],
                                                              "gold_and_terms.pickle",
                                                              comments=["Mary arrived whistling", "Mary tried to like the cat", "cute cat", "likes the cat (incomplete graph)"])

The pickle-building functions can be used to visualise whatever you would like to see.

In [ ]:
# You can run this unittest to see quickly if your operations are working for the three examples given.

import unittest

class TestAMOperations(unittest.TestCase):
    def test_apply(self):
        student_graph = mary_cat_term.evaluate()
        self.assertEqual(student_graph, mary_cat_graph)
    def test_modify(self):
        student_graph = cute_cat_term.evaluate()
        self.assertEqual(student_graph, cute_cat_graph)
    def test_apply_and_modify_together(self):
        student_graph = mary_arrived_whistling_term.evaluate()
        self.assertEqual(student_graph, mary_arrived_whistling_graph)

In [ ]:
# run the test
unittest.main(argv=[''], verbosity=2, exit=False)

## Part 5: Typed AM Algebra

In two questions above, you defined an untyped AM Algebra and a type graph `AMType` class. The next cell defines for you a class for Annotated A-Graphs `ASGraph`, which inherits from SGraph. In addition to its own nodes, edges, etc, it also has a `am_type`, which is a `AMType` object (which you completed above, and which inherits from `Graph`).


In [ ]:
# Provided

class ASGraph(SGraph):
    """
    Implements an Annotated S-graph, as used by a typed Apply-Modify Algebra
    Inherits from SGraph, so it has the S-Graph components like nodes and sources
        as well as an attribute am_type, which is an AMType graph.
    """
    def __init__(self, nodes=None, edges=None, node_labels=None,
                 sources=None, root=None, am_type: AMType or None = None):
        super().__init__(nodes, edges, node_labels, sources, root)
        self.am_type = AMType() if am_type is None else am_type

    @staticmethod
    def from_s_graph_and_type(g: SGraph, t: AMType):
        """
        Generates a new ASGraph from an SGraph and an AMType.
        This is a static method, and can be called without creating an ASGraph instance as follows:
        my_new_as_graph = ASGraph.from_s_graph_and_type(my_s_graph, my_type)
        """
        return ASGraph(g.nodes, g.edges, g.node_labels, g.sources, g.root, t)

    def __repr__(self):
        ret = super().__repr__()
        ret += f"Type:\t{self.am_type}"
        return ret


### QUESTION 4: Typed AM Algebra [10 points]

The next cell defines a `TypedAMAlgebra`, which inherits from `AMAlgebra` above. The initialiser is defined for you. All other functions can be inherited as they are, so only `apply` and `modify` need to be updated. However, you only need to update `apply`. We will leave the `modify` operation unconstrained.

Complete the `apply` method so that it checks whether the types are compatable and returns as `ASGraph` with the correct type. If the types aren't compatable, raise an `AlgebraError` and include a message.

In [ ]:
# %%%%%%%%%%%%%%%% QUESTION 4 %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

class TypedAMAlgebra(AMAlgebra_correct):
    def __init__(self, sources=None, app_only=None, mod_only=None,
                 graph_type=ASGraph, root_source=None):
        super().__init__(sources, app_only, mod_only,
                 graph_type, root_source)

    def apply(self, source: str, head: ASGraph, argument: ASGraph):
        """
        Uses the types of the graphs to check whether Apply is legal.
        # TODO update this docstring
        @return an ASGraph with the correct updated type
        """
        # TODO
        ...


    def modify(self, source: str, head: ASGraph, modifier: ASGraph):
        """
        Implements Modify for ASGraphs without type checking.
        The am_type of the new graph is just the type of he head
        """
        return ASGraph.from_s_graph_and_type(super().modify(source, head, modifier,), head.am_type)

You can test your code here:

In [ ]:
typed_algebra = TypedAMAlgebra(root_source=RT)

typed_constants = {}

for c in constants:
    g = constants[c].function
    if len(g.sources) == 1:
        typed_constants[c] = AlgebraOp(name=c, function=ASGraph.from_s_graph_and_type(g, AMType()))
    elif len(g.sources) == 2:
        sources = [s for s in g.sources if s != typed_algebra.root_source]
        typed_constants[c] = AlgebraOp(name=c, function=ASGraph.from_s_graph_and_type(g, AMType(set(sources))))
    else:
        print("need")
        print(c, g)

typed_constants["tried"] = AlgebraOp(name="tried", function=ASGraph.from_s_graph_and_type(constants["tried"].function, control_type))
typed_constants["like_ctrl"] = AlgebraOp(name="like", function=ASGraph.from_s_graph_and_type(constants["like"].function, control_type))
typed_constants["like"] = AlgebraOp(name="like", function=ASGraph.from_s_graph_and_type(constants["like"].function, AMType({"S", "O"})))


for c in typed_constants:
    print("\n", c)
    print(typed_constants[c].function)

typed_algebra.add_constants(typed_constants)

In [ ]:
like_cat_term = AlgebraTerm(typed_algebra.ops["App_O"], [typed_algebra.make_leaf("like"), typed_algebra.make_leaf("cat")])
like_cat_term.evaluate()

In [ ]:
mary_cat_term = AlgebraTerm(typed_algebra.ops["App_S"], [AlgebraTerm(typed_algebra.ops["App_O"], [typed_algebra.make_leaf("tried"), like_cat_term]), typed_algebra.make_leaf("mary")])
mary_cat_term.evaluate()

## Part 5: Error analysis of AM parser output

### QUESTION 5 [16 points]

Instructions: use Vulcan to display predictions made by an AM Parser on the development set of the Little Prince corpus:

```
python launch_vulcan.py little_prince.pickle
```

You can drag edges around on the dependency tree and nodes and edges on the graphs, if you're having trouble seeing things.

To see the alternatives the parser considered and their probabilities, hold CTRL while mousing over the supertag or edge (for alternative labels) or head index (for alternative heads).

Look through the predictions and answer the following questions. Please write your answers up and hand them in as a PDF.

A. [1 point] Could these parses have come from the Projective approximative decoder discussed in class? Explain why or why not.

B. [4 points] Sentence 7 ("I pondered deeply, then, over the adventures of the jungle") is very close to being correct, but it is not quite right. Answer the following:

1. What's wrong with the graph?
2. Looking at the top alternatives (CTRL-hover), could the parser have gotten it right using alternatives?
3.
    * If so:
        * What are those alternatives?
        * What would the AM dependency tree have looked like instead (describe and/or draw it)
        * How much lower were those probabilities? (just report the probability pairs)
    * If not, why not?

C. [10 points] There are 11 items in the corpus that contain graphs meaning "boa constrictor". To see them all, use the search function. For this question, search the sentence for "constrictor". (Note that there seems to be a bug in the regular expression search, so stick to exact match.)

1. What does the gold subgraph look like that seems to mean "boa constrictor"? Draw it in LaTeX or (clearly) screenshot it and include it in your document.

2. The AM parser makes mistakes. Every time it encounters "boa" and "constrictor", the supertagger predicts graphs for the two words (or no graph) and the edge model predicts operations for putting them together.

    Look through the 11 sentences. How many different ways does the parser build a graph for "boa constrictor"? For each way, provide a subtree of the AM tree (the term or the dependency tree are both fine), and count how many times that subtree is predicted.

    For each subtree, evaluate it to show what graph it builds. If any trees build the same graph, make a note of it.

    Discuss your results, addressing the following: Are there multiple ways the parser builds the same graph? How many times does the parser predict each? How many of the sentences have correct subgraphs for "boa constrictor"? Are there any very weird results making it hard to even decide which subtree to extract? Did you notice anything interesting in the mistakes you observed, did looking at these results tell you anything about the behaviour of the parser, AMR annotation, etc.?

